# CMSC 173 &middot; Midterm Examination &middot; Gradient Descent
### Modules 3 (linear regression), 4 (regularization) and 6 (logistic regression)

**Take-home. 100 points.** Your data is built from **your student number**, so every student has
different numbers. You may use your notes, the slides and the module notebooks.

**Rules for the maths.** Use the course notation: $m$ examples, parameters $\theta_0, \theta_1, \dots$,
learning rate $\alpha$, and sums over examples, $\sum_{i=1}^{m}$. No matrix notation. Write derivations
in the markdown cells marked **Answer**, with every step; a correct number with no working earns
part of the marks only.

**Rules for the code.** NumPy only (no scikit-learn, no `np.linalg`, no `@`). Write gradient descent
yourself. At the end, copy your numbers into the `ANSWERS` cell; that cell is what is graded for
numerical accuracy.

| Part | Question | Points |
|---|---|---|
| A | 1 &middot; The gradient of the squared-error cost | 10 |
| A | 2 &middot; Gradient descent, by hand and in code | 10 |
| A | 3 &middot; When does gradient descent converge? | 20 |
| A | 4 &middot; Ridge regression by gradient descent | 20 |
| A | 5 &middot; Logistic regression by gradient descent | 20 |
| B | 6 &middot; Application: forty condos | 20 |

## Your data
Type your 9-digit student number, then run the next two cells.

In [ ]:
STUDENT_NUMBER = "000000000"   # <- your student number, e.g. "202312345"

In [ ]:
import numpy as np


def _separable_1d(x, y):
    s = y[np.argsort(x)]
    return bool(np.all(np.diff(s) >= 0) or np.all(np.diff(s) <= 0))


def make_data(student_number):
    rng = np.random.default_rng(int(student_number))
    while True:                                                   # Part A: five condos
        area = np.sort(rng.choice(np.arange(20, 130, 10), 5, replace=False)).astype(float)
        b0, b1 = rng.uniform(0.5, 1.5), rng.uniform(0.03, 0.05)
        price = np.round(b0 + b1 * area + rng.normal(0, 0.5, 5), 1)
        cuts = [t for t in np.unique(price)[1:] if 2 <= (price >= t).sum() <= 3
                and not _separable_1d(area, (price >= t).astype(int))]
        if not cuts or price.min() <= 0:
            continue
        threshold = float(cuts[0])
        lab = (price >= threshold).astype(int)
        xs = (area - area.mean()) / area.std()
        w = np.zeros(2)
        for _ in range(10000):                                    # Q5's logistic regression, as asked
            p = 1 / (1 + np.exp(-(w[0] + w[1] * xs)))
            g = np.r_[np.mean(p - lab), np.mean((p - lab) * xs)]
            w -= g
        if np.abs(g).max() < 1e-5 and w[1] >= 0.5:                # converged, and premium rises with area
            break
    n = 40                                                        # Part B: forty condos
    while True:
        c0, c1, c2 = rng.uniform(0.5, 1.5), rng.uniform(0.035, 0.05), rng.uniform(0.02, 0.05)
        A = rng.integers(25, 151, n).astype(float)
        F = rng.integers(2, 41, n).astype(float)
        P = np.round(c0 + c1 * A + c2 * F + rng.normal(0, 1.1, n), 1)
        tB = float(np.round(np.quantile(P[:30], 0.7), 1))
        prem = (P >= tB).astype(int)
        X = np.c_[A, F]
        Xs = (X - X[:30].mean(0)) / X[:30].std(0)
        w = np.zeros(3)
        for _ in range(10000):                                    # Q6's logistic regression, as asked
            p = 1 / (1 + np.exp(-(w[0] + Xs[:30, 0] * w[1] + Xs[:30, 1] * w[2])))
            g = np.r_[np.mean(p - prem[:30]), np.mean((p - prem[:30]) * Xs[:30, 0]),
                      np.mean((p - prem[:30]) * Xs[:30, 1])]
            w -= g
        # keep only data on which those 10,000 steps have converged (so not separable) and whose
        # held-out rows contain both classes
        if np.abs(g).max() < 1e-5 and len(set(prem[30:])) == 2:
            break
    new_unit = (float(rng.integers(40, 141)), float(rng.integers(3, 39)))
    return {
        "area": area, "price": price, "premium_threshold": threshold, "premium": (price >= threshold).astype(int),
        "B_area": A, "B_floor": F, "B_price": P, "B_premium_threshold": tB, "B_premium": prem,
        "new_unit": new_unit,
    }

data = make_data(STUDENT_NUMBER)
x, y, premium = data["area"], data["price"], data["premium"]
m = len(x)
print("Part A: five condos")
print("area (sqm):   ", x)
print("price (PHP M):", y)
print(f"premium = price >= {data['premium_threshold']}:", premium)

---
# Part A &middot; Five condos

Model: $\hat{y}^{(i)} = \theta_0 + \theta_1 x^{(i)}$, with $x$ the area (sqm) and $y$ the price (&#8369; millions).
Cost, as in module 3:
$$J(\theta_0, \theta_1) = \frac{1}{2m} \sum_{i=1}^{m} \left(\theta_0 + \theta_1 x^{(i)} - y^{(i)}\right)^2$$

## Question 1 &middot; The gradient of the squared-error cost (10 points)

**(a)** [6] Derive $\dfrac{\partial J}{\partial \theta_0}$ and $\dfrac{\partial J}{\partial \theta_1}$. Name the rule you use at each step.

**(b)** [4] Evaluate both at $\theta_0 = \theta_1 = 0$ **by hand** for your five condos. Show the sums.

**Answer.** Your answer here.

## Question 2 &middot; Gradient descent, by hand and in code (10 points)

Use $\alpha = 0.0001$ and start at $\theta_0 = \theta_1 = 0$.

**(a)** [4] By hand, take **three** steps of gradient descent, updating $\theta_0$ and $\theta_1$
**simultaneously**. After each step write $\theta_0$, $\theta_1$ and $J$, to 6 significant figures.

**(b)** [3] Write `gd_step(t0, t1, alpha)` that returns the new $(\theta_0, \theta_1)$, and check your table.

**(c)** [3] A common bug updates $\theta_0$ first and then uses the **new** $\theta_0$ to compute
$\theta_1$'s gradient (a *sequential* update). Do one sequential step from $(0, 0)$. What $\theta_1$ does it
give, how far is it from the simultaneous $\theta_1$, and why does the order matter?

**Answer.** (a) your table; (c) your explanation

In [ ]:
def J(t0, t1):
    return np.mean((t0 + t1 * x - y) ** 2) / 2

def gd_step(t0, t1, alpha):
    # your code here
    ...

## Question 3 &middot; When does gradient descent converge? (20 points)

Take the one-parameter model $\hat{y} = \theta_1 x$ (no $\theta_0$). Its cost has a single minimiser
$\theta_1^{*} = \dfrac{\sum_i x^{(i)} y^{(i)}}{\sum_i \left(x^{(i)}\right)^2}$. Let $S = \frac{1}{m}\sum_i \left(x^{(i)}\right)^2$.

**(a)** [8] Show that one gradient-descent step gives
$$\theta_1^{\text{new}} - \theta_1^{*} = (1 - \alpha S)\left(\theta_1 - \theta_1^{*}\right).$$
*Hint:* write the gradient as $S\theta_1 - \frac{1}{m}\sum_i x^{(i)}y^{(i)}$, then use the definition of $\theta_1^*$.

**(b)** [4] Conclude that gradient descent converges for **every** starting point exactly when
$0 < \alpha < 2/S$. Compute $S$ and the bound $2/S$ for your raw areas.

**(c)** [3] Standardise the area, $\tilde{x} = (x - \bar{x})/s$, with $s$ the standard deviation that divides
by $m$ (NumPy's `x.std()`). What are $\tilde{S}$ and the bound now? Why is this the reason module 4
told you to standardise?

**(d)** [5] On the raw areas, run 30 steps from $\theta_1 = 0$ with $\alpha = 0.9 \times (2/S)$, then with
$\alpha = 1.1 \times (2/S)$. Report $|\theta_1 - \theta_1^{*}|$ after 30 steps for each, and check both
against part (a): what should they be, exactly?

**Answer.** Your answer here.

In [ ]:
# your code here

## Question 4 &middot; Ridge regression by gradient descent (20 points)

Use the standardised area $\tilde{x}$ from Question 3 and module 4's ridge cost (the intercept is not penalised):
$$J_{\text{ridge}}(\theta_0, \theta_1) = \frac{1}{2m}\sum_{i=1}^{m}\left(\theta_0 + \theta_1\tilde{x}^{(i)} - y^{(i)}\right)^2 + \lambda\,\theta_1^2$$

**(a)** [6] Derive the update rules and show that the one for $\theta_1$ can be written
$$\theta_1 \leftarrow (1 - 2\alpha\lambda)\,\theta_1 - \alpha\,\frac{1}{m}\sum_{i}\left(\theta_0 + \theta_1\tilde{x}^{(i)} - y^{(i)}\right)\tilde{x}^{(i)}.$$
Why is ridge also called **weight decay**?

**(b)** [6] Set both partial derivatives to zero. Using $\sum_i \tilde{x}^{(i)} = 0$ and $\frac{1}{m}\sum_i (\tilde{x}^{(i)})^2 = 1$, show
$$\theta_0 = \bar{y}, \qquad \theta_1 = \frac{\frac{1}{m}\sum_i \tilde{x}^{(i)} y^{(i)}}{1 + 2\lambda}.$$

**(c)** [5] For $\lambda = 0, 0.1, 0.5, 2$: compute $\theta_1$ from (b), then run 500 steps of gradient descent from
$(0, 0)$ with $\alpha = 1/(1 + 2\lambda)$. Make a table: $\lambda$, closed-form $\theta_1$, gradient-descent $\theta_1$.

**(d)** [3] Repeat Question 3(a)'s argument for $\theta_1$ here. What is the convergence bound on $\alpha$ now,
and why is $\alpha = 1/(1+2\lambda)$ always safe?

**Answer.** Your answer here.

In [ ]:
# your code here

## Question 5 &middot; Logistic regression by gradient descent (20 points)

Classify **premium** ($y = 1$) from the standardised area $\tilde{x}$:
$p^{(i)} = \sigma\!\left(\theta_0 + \theta_1\tilde{x}^{(i)}\right)$ with $\sigma(z) = \dfrac{1}{1 + e^{-z}}$, and module 6's log loss
$$J(\theta_0, \theta_1) = -\frac{1}{m}\sum_{i=1}^{m}\left[y^{(i)}\log p^{(i)} + \left(1 - y^{(i)}\right)\log\left(1 - p^{(i)}\right)\right].$$

**(a)** [5] Show that $\sigma'(z) = \sigma(z)\left(1 - \sigma(z)\right)$.

**(b)** [7] Show that $\dfrac{\partial J}{\partial \theta_1} = \dfrac{1}{m}\sum_i \left(p^{(i)} - y^{(i)}\right)\tilde{x}^{(i)}$ (and the same without $\tilde{x}^{(i)}$ for $\theta_0$).
Which term of the derivation cancels, and why does that make this pairing of sigmoid and log loss convenient?

**(c)** [4] By hand: one step from $(0, 0)$ with $\alpha = 1$. Give both gradients and the new $(\theta_0, \theta_1)$.

**(d)** [4] Run 10,000 steps with $\alpha = 1$. Report $\theta_0$, $\theta_1$, the log loss, the training accuracy
(predict 1 when $p \ge 0.5$), and the decision boundary **in square metres**.

**Answer.** Your answer here.

In [ ]:
# your code here

---
# Part B &middot; Application: forty condos (20 points)

Forty condos with area (sqm), floor, and price. Rows 0&ndash;29 are for **training**; rows 30&ndash;39 are
**held out** and must not be used to fit anything. Use your own gradient-descent code from Part A, extended
to two features: $\hat{y} = \theta_0 + \theta_1 \tilde{a} + \theta_2 \tilde{f}$.

In [ ]:
A, F, P = data["B_area"], data["B_floor"], data["B_price"]
PREM = data["B_premium"]                      # premium = price >= data["B_premium_threshold"]
train, held_out = slice(0, 30), slice(30, 40)
new_area, new_floor = data["new_unit"]
print("premium threshold:", data["B_premium_threshold"], "| new unit:", new_area, "sqm, floor", new_floor)

**(a)** [4] *Choosing $\alpha$ (Question 3).* Compute $S$ for the raw training areas and for the raw training floors,
and the bound $2/S$ for each. Explain why you must standardise before running gradient descent. Standardise both
features **with the training rows' mean and standard deviation** (and apply the same numbers to the held-out rows
and the new unit). Use $\alpha = 1$, half the standardised bound.

**(b)** [4] *Linear regression.* Fit $\theta_0, \theta_1, \theta_2$ with 2,000 steps from zero. Report them and the
held-out mean squared error, $\frac{1}{10}\sum (\hat{y} - y)^2$ over rows 30&ndash;39.

**(c)** [4] *Ridge (Question 4).* Penalise $\lambda(\theta_1^2 + \theta_2^2)$ for $\lambda = 0, 0.1, 1$, each with
$\alpha = 1/(1 + 2\lambda)$ and 2,000 steps. Report the held-out MSE of each and choose $\lambda$. Did ridge help here?
Say why it might or might not, with 30 training rows and two features.

**(d)** [4] *Logistic regression (Question 5).* Classify premium from both standardised features, 10,000 steps,
$\alpha = 1$. Report $\theta_0, \theta_1, \theta_2$ and the held-out accuracy. Then give the **odds ratio for
10 more square metres** at the same floor, $e^{\theta_1 \cdot 10 / s_{\text{area}}}$, and say what it means in a sentence.

**(e)** [4] *The new unit.* Predict its price with your chosen ridge model and its probability of being premium.
In 3&ndash;4 sentences: would you list it as premium, and how far do you trust each number? Name one thing about
this data that limits your trust.

**Answer.** Your answer here.

In [ ]:
# your code here

**Bonus (+5).** In Part B you used $\alpha = 1$, half of the one-feature bound $2$. With **two** standardised
features the bound is lower. Find by experiment (to 2 decimals) the largest $\alpha$ for which Part B(b)'s gradient
descent still converges, then explain it with $r$, the correlation between your standardised training area and floor.
*Hint:* compare your $\alpha$ with $2/(1 + |r|)$.

**Answer.** Your answer here.

---
## Your answers (graded for numerical accuracy)

Fill in every value with the variables from your code, not by retyping digits.

In [ ]:
ANSWERS = {
    "student_number": STUDENT_NUMBER,
    "q1_g0": None, "q1_g1": None,
    "q2_steps": None,                      # [(theta0, theta1, J), (theta0, theta1, J), (theta0, theta1, J)]
    "q2_sequential_theta1": None,
    "q3_S": None, "q3_bound_raw": None, "q3_bound_standardised": None,
    "q3_error_after_30_under": None, "q3_error_after_30_over": None,
    "q4_gd_theta1": None,                  # {0: ..., 0.1: ..., 0.5: ..., 2: ...}
    "q5_step1_theta": None,                # (theta0, theta1)
    "q5_theta": None, "q5_loss": None, "q5_accuracy": None, "q5_boundary_sqm": None,
    "q6_bound_area": None, "q6_bound_floor": None,
    "q6_heldout_mse": None,                # {0: ..., 0.1: ..., 1: ...}
    "q6_logistic_theta": None, "q6_heldout_accuracy": None, "q6_odds_ratio_per_10_sqm": None,
    "q6_new_price": None, "q6_new_p_premium": None,
}
ANSWERS

---
## Submitting

**You can submit only once.** Run every cell first, check your answers and that `STUDENT_NUMBER` is
yours, then run the cell below. It asks for a **submit token**: open
[https://portal.latarak.com/student/submit-token](https://portal.latarak.com/student/submit-token),
sign in, press the button, and paste the token when the cell asks.

Not using Colab? **File &rarr; Download &rarr; Download .ipynb**, then upload it at
[https://portal.latarak.com/course/cmsc173/exam/midterm/submit](https://portal.latarak.com/course/cmsc173/exam/midterm/submit).

In [ ]:
# --- Submit this exam (once) ---------------------------------------------------
import getpass, json, urllib.request, urllib.error

PORTAL, COURSE, EXAM = "https://portal.latarak.com", "cmsc173", "midterm"

try:
    from google.colab import _message
except ImportError:
    raise SystemExit("Not running in Colab. Download this notebook (File > Download > Download .ipynb) "
                     "and upload it at " + PORTAL + "/course/" + COURSE + "/exam/" + EXAM + "/submit")

print("You can submit only once. Student number in this notebook:", STUDENT_NUMBER)
if input("Type SUBMIT to send this notebook as your final answer: ").strip() != "SUBMIT":
    raise SystemExit("Not submitted.")

nb = _message.blocking_request("get_ipynb", timeout_sec=90)["ipynb"]
token = getpass.getpass("Submit token (hidden as you type): ").strip()
req = urllib.request.Request(
    PORTAL + "/api/exams/" + COURSE + "/" + EXAM + "/submit-notebook",
    data=json.dumps({"notebook": nb}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer " + token},
    method="POST",
)
try:
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.load(r)
    print("Submitted", out["exam"], "for", out["student"], "at", out["submitted_at"])
    print(out["message"])
except urllib.error.HTTPError as e:
    print("Not submitted:", json.loads(e.read()).get("error", e.reason))